# Distilling SAM into YOLO`GroundingDINO` locates an object from a text prompt, `SAM` turns that box into amask. Together they label images without ever being trained on them, at roughly asecond per image. That is fine to build a dataset once and far too slow to run ina pipeline, so a YOLO segmentation model trains on their output and does the workafterwards at a few milliseconds per image. `models/yolo_seg_caps.pt` came out ofexactly this.The teachers are wrong sometimes, and a wrong label is worse than a missing one:the student learns the mistake and nothing downstream says so. So labelling andtraining are two steps here, with a look at the dataset in between.

In [ ]:
from pathlib import Pathfrom src.distillation import DistillerDATA = Path("/Users/timothee/Documents/programming/DIHT/data/original_data")OUT = Path("/Users/timothee/Documents/programming/DIHT/data/yolo_dataset")image_paths = sorted(DATA.rglob("*.jpg"))print(f"{len(image_paths)} images")

## 1. Label`labels` is the text prompt, one term per class in class-id order. It is a methodargument, not a constructor one, so the same distiller can label a differentobject without being rebuilt.`mask_threshold` is the one worth tuning: it is SAM's own confidence in the maskit produced. Raising it rejects more and labels less.

In [ ]:
distiller = Distiller(box_threshold=0.4, mask_threshold=0.85)report = distiller.label(image_paths, labels=["bottle cap"], out_dir=OUT, val_ratio=0.2)print(report.summary())

## 2. Look at what came outTwo things to check before training. The rejection rate says how far the promptcarries: a high one usually means the wording, not the images. And `rejected/`holds what the teachers refused, sorted by reason, so a glance says whether theywere right to refuse.`scores.csv` carries the per-image confidences if you would rather sort than look.

In [ ]:
import pandas as pdscores = pd.read_csv(OUT / "scores.csv")print(scores[["detector_best", "mask_best", "instances_found"]].describe().round(3))for folder in sorted((OUT / "rejected").glob("*")):    print(f"{folder.name}: {len(list(folder.glob('*')))} images")

In [ ]:
import cv2import numpy as npfrom PIL import Imagedef show_label(image_path: Path, label_path: Path, margin: float = 0.35):    """Draw a YOLO polygon over its image, cropped to the instance."""    image = np.array(Image.open(image_path).convert("RGB"))    height, width = image.shape[:2]    for line in label_path.read_text().strip().split("\n"):        points = np.array(list(map(float, line.split()[1:]))).reshape(-1, 2)        points *= [width, height]        points = points.astype(np.int32)        filled = image.copy()        cv2.fillPoly(filled, [points], (0, 255, 0))        image = cv2.addWeighted(filled, 0.45, image, 0.55, 0)        cv2.polylines(image, [points], True, (255, 0, 0), 14)    x, y, w, h = cv2.boundingRect(points)    side = max(w, h) + 2 * int(margin * max(w, h))    x0, y0 = max(0, x + w // 2 - side // 2), max(0, y + h // 2 - side // 2)    return Image.fromarray(image[y0:y0 + side, x0:x0 + side]).resize((400, 400))sample = sorted((OUT / "images" / "train").glob("*.jpg"))[0]show_label(sample, OUT / "labels" / "train" / sample.with_suffix(".txt").name)

Stored images already have their EXIF rotation applied, so `Image.open` aboveneeds no `exif_transpose`. Copying the originals untouched would leave theorientation to the reader, and readers disagree — cv2 applies the tag, PIL doesnot — while the polygons are normalised against the rotated frame. A viewer thatskipped the tag would draw a mask that misses the object entirely.## 3. Train the studentA pass-through to ultralytics, which owns training. `data.yaml` and the manifestnaming the prompt, both teachers and the thresholds sit next to the dataset.

In [ ]:
print((OUT / "manifest.json").read_text())

In [ ]:
distiller.train(OUT / "data.yaml", model="yolov8n-seg.pt", epochs=50, imgsz=640)

The trained weights land under `runs/segment/train*/weights/best.pt`. Point`YOLOCustomCrop` at them and the teachers are no longer needed:```pythonfrom src.preprocess import YOLOCustomCropcrop = YOLOCustomCrop("runs/segment/train/weights/best.pt")cropped = crop(Image.open(some_photo))```On this dataset the student was worth 7.4 points of retrieval recall through thetighter crop alone, and 6.0 more through the background removal, measured onMobileNetV3 over 40 paired draws.